# **Baseband Data Classes (BDC)**
## Learning Objectives
This notebook will teach you all about BDC. There are various objects associated with it, and it can be used to handle data in several ways, from just checking a file header to iterating through a full data of data. 

## Prerequisites
You must have albatros_analysis installed, and have some familiarity with how data is saved and organized. More information can be found on the [Data Structure Page](/pipeline/data_structure/).

In [3]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from os import path
import sys
sys.path.insert(0, "/home/thomasb/")

# Baseband Data Classes script
from albatros_analysis.src.correlations import baseband_data_classes as bdc

# Baseband utils script
from albatros_analysis.src.utils import baseband_utils as butils
from albatros_analysis.scripts.xcorr import helper as hxc

BDC is using numpy


## 1. A Baseband Object
The first and most basic thing we can do with BDC is create a **Baseband Object**, which lets you take a single file and treat it. Let's consider some file somewhere, and define some useful parameters that we know. Recall that spectra (time samples) have a time resolution of about 16 microseconds and frequency resolution of about 60 kHz.

In [1]:
file_name = "/scratch/mohanagr/summer_2025/baseband/mars1/17532/1753209977.raw"
T_SPECTRA = 4096/250e6
dfreq = 250e6/4096
print('Time resolution:', T_SPECTRA*1e6, 'microsec')
print('Frequency resolution', dfreq/1e3, 'kHz')

Time resolution: 16.384 microsec
Frequency resolution 61.03515625 kHz


We look at this file through the Baseband Object, which allows us to easily access some information about the file. Two examples (of many possible options) are shown below.

In [ ]:
#create the object
file1 = bdc.Baseband(file_name, readlen=-1, force_cpu=False, verbose=True)
#look at spectrum number corresponding to each time sample
print('spectrum numbers:', file1.spec_num)
#look at whether it's 1 bit or 4 bit
print('in bit mode:', file1.bit_mode)

took 0.061 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753209977.raw
spectrum numbers [601090400 601090408 601090416 ... 604369064 604369072 604369080]
in bit mode: 1


The most useful function here lets you see the whole file header

In [12]:
file1.print_header()

Header Bytes = 1304. Bytes per packet = 1220. Channel length = 304. Spectra per packet: 8. Bit mode: 1. Total packets = 409836. Read packets = -1. Have trimble = 1. Channels: [  60   61   62   63   64   65   66   67   68   69   70   71   72   73
   74   75   76   77   78   79   80   81   82   83   84   85   86   87
   88   89   90   91   92   93   94   95   96   97   98   99  100  101
  102  103  104  105  106  107  108  109  110  111  112  113  114  115
  116  117  118  119  120  121  122  123  124  125  126  127  128  129
  130  131  132  133  134  135  136  137  138  139  140  141  142  143
  144  145  146  147  148  149  150  151  152  153  154  155  156  157
  158  159  196  197  198  199  200  201  202  203  204  205  206  207
  208  209  210  211  212  213  214  215  216  217  218  219  220  221
  222  223  224  225  226  227  228  229  230  231  232  233  234  235
  236  237  238  239  240  241  242  243  244  245  246  247  248  249
  250  251  252  253  254  255  256  257  25

## 2. Baseband File Iterator
Other than checking the file header, the Baseband object is, as a standalone, not particularly useful when you want to examine long stretches of data. It's used as the foundation for a second object, the **Baseband File Iterator** (BFI). This lets you move through many files, and also handles dropped spectra. It splits long periods of data into chunks of a desired number of spectra, and loads file data into them iteratively. To get started and see how it works, let's define some more useful parameters. 

In [13]:
#what time intervals do I want to look at
start_ts = 1753210000
end_ts =   1753220000

#where does all the data live (the root dir)
path_data = "/scratch/mohanagr/summer_2025/baseband/mars1"

#how many spectra per chunk
chunk_size = 1000000
#how many chunks
nchunks = 20

#what channel indices do I want to look at
chanstart = 1834
chanend = 1852

Since we're now working with time and not discrete files, we need to figure out two main things. Which files contain all the data we want, and where within the first file our starting timestamp corresponds to. Files are almost a minute long, so figuring out the starting spot in the first file is not trivial. The parameter `idx` is the spectrum index in the first file for which we say the starting timestamp corresponds to. We have a useful function in `utils/baseband_utils` that finds these two parameters efficiently for some given antenna and time interval.

In [ ]:
try:
    files, idx = butils.get_init_info(start_ts, end_ts, path_data)
except Exception as e:
    print(e)
    print(f"WARNING: MISTAKE IN FILE CHECKER!!")
    sys.exit()

print('first file', files[0])
print('idx within first file:', idx)
print('last file', files[-1])

1753209977 1753219967
first file /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753209977.raw
idx within first file: 1403809
last file /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753219967.raw


As a side note, the order 1 second timestamping errors we talk about in the [Pipeline Overview](/pipeline/overview/) comes from the fact that we do not know `idx` accurately. This comes from the fact that files do not contain reliable information about when their data was actually recorded.

The next step is to determine the channel indices we want within the data. The channel numbers we use (integer multiples of the channel resolution) may not be the same channel indices within the file.

In [5]:
# Set up the number of channels we look through
print("Setting Antenna as BFI Objects", '\n')
channels = np.asarray(bdc.get_header(files[0])["channels"],dtype='int64')
chanstart_idx = np.where(channels == chanstart)[0][0]
chanend_idx = np.where(channels == chanend)[0][0]
nchans = chanend_idx - chanstart_idx

print('channel start', chanstart, 'with file index', chanstart_idx)
print('channel end', chanend, 'with file index', chanend_idx)

Setting Antenna as BFI Objects 

Not reading any data
channel start 1834 with file index 284
channel end 1852 with file index 302


Now we define an antenna object. This is the interface that lets us interact with and parse through data effectively and efficiently

In [6]:
nchans = chanend - chanstart
ant = bdc.BasebandFileIterator(
    files,
    0,
    idx,
    chunk_size,
    nchunks,
    chanstart=chanstart_idx,
    chanend=chanend_idx,
    type="float",
)

print('BFI object type', type(ant))
print('acclen', ant.acclen)
print('nchunks', ant.nchunks)

ACCLEN RECEIVED IS 1000000
took 0.438 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753209977.raw
START SPECNUM IS 602494209 obj start at 601090400
submitted next file queue job /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753210031.raw
BFI object type <class 'albatros_analysis.src.correlations.baseband_data_classes.BasebandFileIterator'>
acclen 1000000
nchunks 20


took 0.451 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753210031.raw
took 0.476 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753210084.raw
took 0.466 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753210136.raw
took 0.438 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753210191.raw
took 0.402 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753210245.raw
took 0.418 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753210300.raw
took 0.406 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753210353.raw


We want to find the starting spectrum number for our data. Spectra are a proxy for time, so we like knowing what spectrum number we start out at.

In [7]:
start_specnum = ant.spec_num_start
print('Initial Specnum:', start_specnum)

Initial Specnum: 602494209


Now we can iterate through the chunks. This is the method we use to read through data when you want to look at data individually, or want to parse through lots of data in a row without demolishing your ram, and while being able to deal with holes in your data.

In [8]:
for chunkidx, chunk in enumerate(ant):
    print(f'\n----- Chunk {chunkidx}/{nchunks-1} ------')

    data = cp.zeros((chunk_size, nchans), dtype="complex64") #remember that BDC returns complex64. wanna do phase-centering in 128.

    print('CHUNK IDX', chunkidx)
    print('NCHUNKS', nchunks)
    if chunkidx == nchunks:
        print('offbyone error!!')
        sys.exit()

    # we want to check how much of the data is present. if it's too much, we skip the chunk
    perc_missing = (1 - len(chunk["specnums"]) / chunk_size) * 100
    if perc_missing > 10:
        print(f'big problem! plenty of data missing {perc_missing}. abort!')
        if chunkidx == 0: #logic here is that maybe after a reboot the data is corrupted or something: will let chunk 1 free
            continue
        sys.exit()  #otherwise we will need to check what's going on. may be a sign of something ystemically wrong in data

    # want the spectrum indices for the entire chunk
    spec_idxs = chunk['specnums']-(start_specnum + chunk_size*chunkidx)
    print('ant start idx', chunk['specnums'][0])

    # make the chunk continuous, and write it to the data array
    bdc.make_continuous_gpu(cp.asarray(chunk['pol0']),
                            spec_idxs,
                            np.arange(nchans),
                            chunk_size,
                            nchans=nchans, 
                            out=data)

    # then you can do whatever with your data, chunk by chunk
    print('data shape', data.shape, 'percentage missing', perc_missing)

print shape of my pols 1000000 18

----- Chunk 0/19 ------
CHUNK IDX 0
NCHUNKS 20
ant start idx 602494209
data shape (1000000, 18) percentage missing 0.0
print shape of my pols 874879 18
submitted next file queue job /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753210084.raw
print shape of my pols 125121 18

----- Chunk 1/19 ------
CHUNK IDX 1
NCHUNKS 20
ant start idx 603494209
data shape (1000000, 18) percentage missing 0.0
print shape of my pols 1000000 18

----- Chunk 2/19 ------
CHUNK IDX 2
NCHUNKS 20
ant start idx 604494209
data shape (1000000, 18) percentage missing 0.0
print shape of my pols 1000000 18

----- Chunk 3/19 ------
CHUNK IDX 3
NCHUNKS 20
ant start idx 605494209
data shape (1000000, 18) percentage missing 0.0
print shape of my pols 1000000 18

----- Chunk 4/19 ------
CHUNK IDX 4
NCHUNKS 20
ant start idx 606494209
data shape (1000000, 18) percentage missing 0.0
print shape of my pols 153567 18
submitted next file queue job /scratch/mohanagr/summer_2025/baseband/

## 3. Multiple Antennas

In the case that you want two (or more!) antenna, and have access to some spectrum number corrections, we load things up slightly differently. Spectrum number offsets are always with respect to the first antenna (usually MARS1), and it has zero offset with respect to itself. From here, it's all the same, with one object for each individual antenna data stream.

In [ ]:
specnum_offsets = [0, -1884333]
paths = ["/scratch/mohanagr/summer_2025/baseband/mars1", 
         "/scratch/mohanagr/summer_2025/baseband/mars2"]

idxs, files = hxc.get_init_info_all_ant(start_ts, end_ts, specnum_offsets, paths)

1753209977 1753219967
Using delta of 64.0 seconds to check for gaps between files
took 0.447 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753209977.raw
1753210000 1753219991
Using delta of 64.0 seconds to check for gaps between files
took 0.651 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars2/17532/1753210000.raw
specnums [602494209, 604335912]
spec offsets [       0 -1884333]

PROCESSING ANTENNA 0
initial offset wrt ref ant 0
idxs before correction 1403809 1403809
spec offset 0 init offset 0
correction 0
after correction 1403809 1403809

PROCESSING ANTENNA 1
initial offset wrt ref ant -1841703
idxs before correction 1403809 0
spec offset -1884333 init offset -1841703
correction -42630
after correction 1403809 42630


## Further Notes
This notebook is focused on the functionalities of BDC, so if you intend to use it, looking at the script and its various functions more carefully is always a good idea. It can be found in the [Correlations API Page](/api/correlations/).